In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 64.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 132.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 27.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit]2m6/7 [qiskit]ne]
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 69.6 MB/s  0:00:00m0:00:01
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=0b6f1c3c9c559ce2b974fff88b2bab602eca395a2862020dfbcc0638a71ad964
  Stored in direct

In [9]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 

B0_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]

B1_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]

def random():
    q = QuantumCircuit(1) 
    q.h(0) 
    q.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    return counts.get("1",0)

def oneThird():
    theta = 2 * math.acos(math.sqrt(1/3))
    q = QuantumCircuit(1)
    q.ry(theta, 0)
    q.measure_all()
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    counts = result.get_counts(compiled)
    return counts.get("1", 0)
    
def randomChoice():
    if oneThird() == 0:
        return 1
    else:
        return 2 + random()

def average(c,n): 
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=n)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    count00 = counts.get("00",0) 
    count01 = counts.get("01",0) 
    count10 = counts.get("10",0) 
    count11 = counts.get("11",0) 
    return (count00 - count01 - count10 + count11) / n 

def entangledPair():
    q = QuantumCircuit(2) 
    q.h(0)
    q.cx(0,1)
    q.x(0)
    q.z(1)
    return q

N = 100
runs = int(9*N/2)

alice_operators = []
bob_operators = []
alice_bits = []
bob_bits = []

for i in range(runs):
    Ai = randomChoice()
    Bj = randomChoice()
    alice_operators.append(Ai)
    bob_operators.append(Bj)
    q = entangledPair()
    
    #X
    if Ai == 1:
        q.h(0)
    #W    
    elif Ai == 2:     
        q.unitary(B0_transform_matrix, [0])
    #Z   
    elif Ai == 3:     
        pass
    #W
    if Bj == 1:       
        q.unitary(B0_transform_matrix, [1])
    #Z   
    elif Bj == 2:     
        pass
    #V    
    elif Bj == 3:     
        q.unitary(B1_transform_matrix, [1])

    q.measure_all()
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    results = list(counts.keys())[0]
    alice_bits.append(int(results[1]))
    bob_bits.append(int(results[0]))

shared_key = []

for i in range(runs):
    Ai = alice_operators[i]
    Bj = bob_operators[i]

    if (Ai == 2 and Bj == 1) or (Ai == 3 and Bj == 2):
        bob_bit = 1 - bob_bits[i]
        shared_key.append(alice_bits[i])

circuit_A0_B0 = entangledPair() 
# Don't transform qubit 0
# Transform qubit 1 by B0_transform
circuit_A0_B0.unitary(B0_transform_matrix,[1])
# Measure both qubits
circuit_A0_B0.measure_all()
A0_B0_average = average(circuit_A0_B0,N) 
#print(A0_B0_average)

# Construct the circuit for measuring A0 B1, simulate it and calculate the average result
circuit_A0_B1 = entangledPair() 
# Don't transform qubit 0
# Transform qubit 1 by B1_transform
circuit_A0_B1.unitary(B1_transform_matrix,[1])
# Measure both qubits
circuit_A0_B1.measure_all()
A0_B1_average = average(circuit_A0_B1,N) 
#print(A0_B1_average)

# Construct the circuit for measuring A1 B0, simulate it and calculate the average result
circuit_A1_B0 = entangledPair() 
# Transform qubit 0 by H
circuit_A1_B0.h(0)
# Transform qubit 1 by B0_transform
circuit_A1_B0.unitary(B0_transform_matrix,[1])
# Measure both qubits
circuit_A1_B0.measure_all()
A1_B0_average = average(circuit_A1_B0,N) 
#print(A1_B0_average)

# Construct the circuit for measuring A1 B1, simulate it and calculate the average result
circuit_A1_B1 = entangledPair() 
# Transform qubit 0 by H
circuit_A1_B1.h(0)
# Transform qubit 1 by B0_transform
circuit_A1_B1.unitary(B1_transform_matrix,[1])
# Measure both qubits
circuit_A1_B1.measure_all()
A1_B1_average = average(circuit_A1_B1,N)
#print(A1_B1_average)

# The final calculuation
result = abs(A0_B0_average + A0_B1_average + A1_B0_average - A1_B1_average)

if result < 2:
    print(f"Attacker detected, |S| = {result:.3f}")
else:
    print(f"No attacker detected, |S| = {result:.3f}")
print (f"Shared key: {shared_key}")

No attacker detected, |S| = 2.840
Shared key: [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0]
